In [13]:
import pandas as pd

# Read CSV from the same directory
df = pd.read_csv('train.csv')

# Renaming columns and convert all column names to lowercase
df.columns = df.columns.str.lower()
df = df.rename(columns={'id': 'customer_id', 'vintage': 'tenure_days'})

# Convert float columns to Nullable Integers
df['region_code'] = df['region_code'].astype('Int64')
df['policy_sales_channel'] = df['policy_sales_channel'].astype('Int64')

# Map 'Yes'/'No' in vehicle_damage to 1/0 integers to match the other binary columns
yes_no_to_one_zero = {'Yes': 1, 'No': 0}
df['vehicle_damage'] = df['vehicle_damage'].map(yes_no_to_one_zero).astype('int8')

# Ensure existing 1/0 columns are stored as compact 8-bit integers
df['driving_license'] = df['driving_license'].astype('int8')
df['previously_insured'] = df['previously_insured'].astype('int8')
df['response'] = df['response'].astype('int8')


# Define the exact integer mapping schema (1, 2, 3)
vehicle_age_mapping = {
    '< 1 Year': 1,
    '1-2 Year': 2,
    '> 2 Years': 3
}
# Create the new integer column 'vehicle_age_order'
df['vehicle_age_order'] = df['vehicle_age'].map(vehicle_age_mapping)

# Slice vertically into 3 DataFrames
dim_customers = df[['customer_id', 'gender', 'age', 'driving_license', 'region_code']].copy()
dim_vehicles = df[['customer_id', 'vehicle_age', 'vehicle_age_order', 'vehicle_damage', 'previously_insured']].copy()
fact_cross_sell_campaigns = df[['customer_id', 'annual_premium', 'policy_sales_channel', 'tenure_days', 'response']].copy()

# Generate sequential ID starting from 1 and insert at index 0 (the first column)
dim_vehicles.insert(
    loc=0,
    column='vehicle_id',
    value=range(1, len(dim_vehicles) + 1)
)
fact_cross_sell_campaigns.insert(
    loc=0,
    column='campaign_id',
    value=range(1, len(fact_cross_sell_campaigns) + 1)
)

# Shows the first 5 rows of the data
print(df.head())

# Sanity check: Ensure that the number of rows in the fact table matches the original DataFrame
print('Row counts =',len(dim_customers))
print('Row counts =',len(dim_vehicles))
print('Row counts =',len(fact_cross_sell_campaigns))
assert dim_customers['customer_id'].is_unique, "Duplicate customer_id found!"
assert dim_vehicles['vehicle_age_order'].isnull().sum() == 0, "Unmapped vehicle_age found!"

#Export the DataFrames to CSV files
dim_customers.to_csv('dim_customers.csv', index=False)
dim_vehicles.to_csv('dim_vehicles.csv', index=False)
fact_cross_sell_campaigns.to_csv('fact_cross_sell_campaigns.csv', index=False)


   customer_id  gender  age  driving_license  region_code  previously_insured  \
0            1    Male   44                1           28                   0   
1            2    Male   76                1            3                   0   
2            3    Male   47                1           28                   0   
3            4    Male   21                1           11                   1   
4            5  Female   29                1           41                   1   

  vehicle_age  vehicle_damage  annual_premium  policy_sales_channel  \
0   > 2 Years               1         40454.0                    26   
1    1-2 Year               0         33536.0                    26   
2   > 2 Years               1         38294.0                    26   
3    < 1 Year               0         28619.0                   152   
4    < 1 Year               0         27496.0                   152   

   tenure_days  response  vehicle_age_order  
0          217         1                